# SORTARA — Training & Evaluasi Model Klasifikasi Sampah (MobileNetV2)

**PIC:** Adit (Aditya Rafaul) & Irfan Dwiki  
**Arsitektur:** CNN Transfer Learning (`MobileNetV2`, input `224x224` RGB)  
**Enam Kategori Kelas (`labels.json`):** `0: Kaca`, `1: Kertas`, `2: Logam`, `3: Organik`, `4: Plastik`, `5: Residu`

---
## Struktur Folder Dataset yang Diharapkan
Siapkan dataset dengan 6 subfolder (masing-masing target awal 200–500 foto):
```text
dataset_raw/
├── Kaca/
├── Kertas/
├── Logam/
├── Organik/
├── Plastik/
└── Residu/
```

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ["Kaca", "Kertas", "Logam", "Organik", "Plastik", "Residu"]

tf.keras.utils.set_random_seed(SEED)
print("TensorFlow version:", tf.__version__)

## 1. Pembagian Dataset Stratified (70% Train / 15% Validation / 15% Test)
Memastikan tidak ada foto yang bocor (*data leakage*) antara training, validation, dan testing.

In [ ]:
DATASET_DIR = Path("../data/dataset_raw")

if not DATASET_DIR.exists():
    import sys
    sys.path.insert(0, str(Path("../scripts").resolve()))
    from prepare_dataset import prepare_sortara_dataset
    prepare_sortara_dataset()

all_filepaths = []
all_labels = []

if DATASET_DIR.exists():
    for idx, cls_name in enumerate(CLASS_NAMES):
        cls_dir = DATASET_DIR / cls_name
        if not cls_dir.exists():
            raise FileNotFoundError(f"Folder kelas tidak ditemukan: {cls_dir}")
        files = [
            str(p)
            for p in cls_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        ]
        all_filepaths.extend(files)
        all_labels.extend([idx] * len(files))
        print(f"{cls_name}: {len(files)} gambar")

    # Split 70% Train, 30% Temp
    X_train, X_temp, y_train, y_temp = train_test_split(
        all_filepaths, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED
    )
    # Split 30% Temp menjadi 15% Val dan 15% Test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
    )
    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
else:
    print("Folder dataset_raw belum ditemukan. Silakan mount Google Drive atau arahkan DATASET_DIR.")

## 2. Pipeline `tf.data` & Preprocessing Konsisten dengan Aplikasi Streamlit
Menggunakan `(pixel / 127.5) - 1.0` (`tf.keras.applications.mobilenet_v2.preprocess_input`).

In [ ]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.08),
        tf.keras.layers.RandomZoom(0.10),
    ],
    name="light_augmentation",
)


def load_and_preprocess_image(path: tf.Tensor, label: tf.Tensor, training: bool = False):
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    if training:
        img = data_augmentation(img, training=True)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(tf.cast(img, tf.float32))
    return img, tf.one_hot(label, depth=len(CLASS_NAMES))


def make_dataset(paths, labels, training: bool = False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(
        lambda p, y: load_and_preprocess_image(p, y, training=training),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 3. Arsitektur Model MobileNetV2 Transfer Learning

In [ ]:
def build_sortara_model(num_classes: int = 6) -> tuple[tf.keras.Model, tf.keras.Model]:
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False  # Tahap 1: Bekukan feature extractor

    inputs = tf.keras.Input(shape=(224, 224, 3), name="input_rgb_normalized")
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="waste_class_probs")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="sortara_mobilenetv2")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base_model

## 4. Training Model, Evaluasi Test Set, & Ekspor `sortara_model.keras` + `labels.json`

In [ ]:
if DATASET_DIR.exists() and len(all_filepaths) > 0:
    train_ds = make_dataset(X_train, y_train, training=True)
    val_ds = make_dataset(X_val, y_val, training=False)
    test_ds = make_dataset(X_test, y_test, training=False)

    model, base_model = build_sortara_model(len(CLASS_NAMES))
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    ]

    history = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

    # Evaluasi pada Test Set terpisah (Bagian 9.4 PRD)
    y_prob = model.predict(test_ds)
    y_pred = np.argmax(y_prob, axis=1)
    print("Overall Test Accuracy:", round(float(accuracy_score(y_test, y_pred)), 4))
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix:\n", cm)

    # Simpan artefak model dan kontrak label untuk aplikasi Streamlit
    out_dir = Path("../models")
    out_dir.mkdir(parents=True, exist_ok=True)
    model.save(out_dir / "sortara_model.keras")
    labels_map = {str(i): cls_name for i, cls_name in enumerate(CLASS_NAMES)}
    (out_dir / "labels.json").write_text(json.dumps(labels_map, indent=2), encoding="utf-8")
    print("Model berhasil disimpan ke ../models/sortara_model.keras")